In [1]:
"""
stage7_final_evaluation.py
============================================================================
Kaggle Notebook. GPU helpful (inference only), CPU workable.

STAGE 7 — the final RQ-D answer. Evaluates the four Stage 6 arms on the
OFFICIAL TEST SPLIT, which no part of the pipeline has ever touched.

ARMS (all trained with identical pos_weight, cap 20, from real images only):
  none            real images only                       -- the reference
  all             real + every reconstruction
  filtered        real + low-variance reconstructions
  random_matched  real + random reconstructions, size-matched to `filtered`

COMPARISONS, in order of how much they tell you:
  1. filtered vs random_matched  -- THE causal test. Same dataset size, only
                                    the selection criterion differs, so a
                                    difference isolates uncertainty filtering.
  2. all vs none                 -- does augmentation help at all?
  3. filtered vs all             -- confounded by dataset size; report with
                                    that caveat.
  4. random_matched vs none      -- effect of simply adding FEWER images.

METRICS
  - per-class AUROC and AUPRC, always reported against the prevalence
    baseline. AUPRC must be read against prevalence, never against 0.5: at
    1.5% prevalence an AUPRC of 0.05 is 3x better than random, not a failure.
  - per-class precision / recall / F1 at a threshold chosen on VALIDATION
    (never on test -- choosing on test would leak).
  - macro-average, reported but de-emphasised: with one class near 52%
    prevalence it drowns out the rare classes RQ-D most concerns.

STATISTICS
  - DeLong's paired test. All four models score the SAME test images, so
    their AUCs are correlated; a naive z-test that ignores that covariance
    is invalid. DeLong uses the Mann-Whitney representation of AUC to model
    the covariance explicitly (DeLong et al. 1988; fast algorithm from
    Sun & Xu 2014).
  - Stratified bootstrap CI on the AUC DIFFERENCE (B=2000), which covers the
    small-sample / heavy-tail cases where DeLong's asymptotics are weakest.
    Rare classes here have few test positives, so this matters.
  - McNemar's test on binarised predictions -- checks whether the models'
    DECISIONS differ, not just their rankings.
  - Benjamini-Hochberg correction across the whole comparison family
    (4 pairs x 5 classes = 20+ tests). Raw and adjusted p both reported.

KNOWN LIMITATION, state it in the thesis: one training run per arm means
there is no estimate of seed-to-seed training variance. The bootstrap CIs
capture test-set sampling variance only. A difference smaller than typical
seed variance cannot be distinguished from training noise.
"""

# ============================================================================
# CELL 0 — Imports
# ============================================================================
try:
    import h5py
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "h5py"], check=True)
    raise SystemExit("h5py installed. RESTART THE KERNEL, then re-run this cell.")

import itertools
import warnings
from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                              precision_recall_curve, precision_score,
                              recall_score, f1_score)
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import mcnemar
from tqdm.auto import tqdm


# ============================================================================
# CELL 1 — CONFIG (must mirror Stage 6)
# ============================================================================
H5_PATH = "/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_512.h5"
METADATA_CSV_PATH = "/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_metadata.csv"

TARGET_CLASSES = ["Pneumothorax", "Consolidation", "Nodule/Mass", "Cardiomegaly", "Atelectasis"]

CHECKPOINT_ROOT = "/kaggle/input/datasets/kartikichandratre/densenet-posw-models/stage6_models"
RUN_TAG = "weighted"                 # matches Stage 6's RUN_TAG
ARMS = ["none", "all", "filtered", "random_matched"]

OUTPUT_DIR = "/kaggle/working/stage7_results"

# Every directory that received GENERATED maps (Stage 1 and Stage 4). Used
# only by the contamination check, which asserts no test image was ever run
# through the pipeline. List everything -- a missed directory means an
# unverified claim, not a passing one.
GENERATED_MAP_DIRS = [
    "/kaggle/input/datasets/pgc17ms072/pneumothorax-uncertainity-train-control",
    "/kaggle/input/datasets/kartikichandratre/atelectasis-uncertainity-train-eta1-n20-g1",
    "/kaggle/input/datasets/pgc17ms072/nodule-uncertainity-train-eta1-n20-g1",
    "/kaggle/input/datasets/pgc17ms072/consolidation-gen-train",
    "/kaggle/input/datasets/pgc17ms072/cardiomegaly-uncertainity-train-eta1-n20-g1-sess0",
    "/kaggle/input/datasets/pgc17ms072/cardiomegaly-unceratainty-train-eta1-n20-g1-sess1",
    "/kaggle/input/datasets/kartikichandratre/cardiomelgaly-uncertainity-sess2",
    "/kaggle/input/datasets/kartikichandratre/stage4-mcd-model-gradcam-uncertinty-maps/stage4_mc_dropout_maps",
    "/kaggle/input/datasets/kartikichandratre/stage4-mcd-logits-uncertainty-maps/stage4_mc_dropout_maps"
    # "/kaggle/input/datasets/pgc17ms072/cardiomegaly-uncertainity-train-eta1-n20-g1-sess0",
    # ... add all Stage 1 + Stage 4 output dirs
]

MODEL_INPUT_SIZE = 512
DROP_RATE = 0.3
BATCH_SIZE = 8
NUM_WORKERS = 4
USE_AMP = True

# Split reproduction — MUST match Stage 6 or the val-chosen thresholds are wrong.
RANDOM_SEED = 42
VAL_FRACTION = 0.15

N_BOOTSTRAP = 2000
ALPHA = 0.05

# The comparisons to test, ordered by how informative they are (see header).
COMPARISON_PAIRS = [
    ("filtered", "random_matched"),
    ("all", "none"),
    ("filtered", "all"),
    ("random_matched", "none"),
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(RANDOM_SEED)


# ============================================================================
# CELL 2 — Contamination check (run FIRST; everything else is void if it fails)
# ============================================================================
def assert_no_test_contamination(metadata, generated_dirs):
    """
    The test split must never have passed through the diffusion or MC-Dropout
    pipeline. Verify rather than assume -- this is the evidence for the
    methods section, and an examiner may ask for it.
    """
    test_ids = set(metadata[metadata["split"] == "test"]["image_id"].astype(str))
    generated = set()
    checked, missing = [], []

    for d in generated_dirs:
        p = Path(d)
        if not p.exists():
            missing.append(d)
            continue
        checked.append(d)
        for f in sorted(p.glob("*.npz")):
            with np.load(f) as data:
                generated |= {k[: -len("__variance")] for k in data.files
                              if k.endswith("__variance")}

    overlap = test_ids & generated
    print(f"  checked {len(checked)} directory(ies); {len(generated)} generated image_ids")
    if missing:
        warnings.warn(
            f"{len(missing)} configured directory(ies) not found and therefore "
            f"NOT checked: {missing}. The contamination check is only as complete "
            f"as GENERATED_MAP_DIRS — an unchecked directory is an unverified claim."
        )
    if overlap:
        raise RuntimeError(
            f"CONTAMINATION: {len(overlap)} test image(s) appear in generated "
            f"maps: {sorted(overlap)[:10]}. Every Stage 7 result would be "
            f"invalid. Investigate before proceeding."
        )
    print(f"  CLEAN: 0 of {len(test_ids)} test images appear in any generated map.")
    return len(test_ids)


# ============================================================================
# CELL 3 — Data + model
# ============================================================================
class EvalDataset(Dataset):
    def __init__(self, image_ids, labels_by_id, h5_path, target_classes, size=MODEL_INPUT_SIZE):
        self.image_ids, self.labels_by_id = image_ids, labels_by_id
        self.h5_path, self.target_classes, self.size = h5_path, target_classes, size
        self._h5 = None

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")
        image_id = self.image_ids[idx]
        arr = self._h5[image_id][()].astype(np.float32)
        t = torch.from_numpy(arr)[None, None]
        if t.shape[-1] != self.size:
            t = F.interpolate(t, size=(self.size, self.size), mode="bilinear", align_corners=False)
        labels = self.labels_by_id.get(image_id, [])
        target = torch.zeros(len(self.target_classes), dtype=torch.float32)
        for i, cls in enumerate(self.target_classes):
            if cls in labels:
                target[i] = 1.0
        return t.squeeze(0).repeat(3, 1, 1), target


def build_densenet121(num_classes, drop_rate=DROP_RATE):
    model = models.densenet121(weights=None, drop_rate=drop_rate)
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(nn.Dropout(p=drop_rate), nn.Linear(in_features, num_classes))
    return model


@torch.no_grad()
def predict(model, loader):
    """Deterministic inference: dropout OFF. Not MC-Dropout -- that is Stage 4."""
    model.eval()
    amp = bool(USE_AMP and DEVICE.type == "cuda")
    probs, targets = [], []
    for images, y in tqdm(loader, desc="    inference", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", dtype=torch.float16, enabled=amp):
            logits = model(images)
        probs.append(torch.sigmoid(logits.float()).cpu().numpy())
        targets.append(y.numpy())
    return np.concatenate(probs), np.concatenate(targets)


def load_arm(arm):
    ckpt = Path(CHECKPOINT_ROOT) / f"{arm}_{RUN_TAG}" / "best.pt"
    if not ckpt.exists():
        raise FileNotFoundError(
            f"No checkpoint at {ckpt}. Expected Stage 6 output for arm '{arm}' "
            f"with RUN_TAG='{RUN_TAG}'. If Stage 6 ran in an earlier session, "
            f"re-attach its output as a Kaggle Dataset and update CHECKPOINT_ROOT."
        )
    state = torch.load(ckpt, map_location=DEVICE)
    n_ckpt = state["classifier.1.weight"].shape[0]
    if n_ckpt != len(TARGET_CLASSES):
        raise ValueError(
            f"{ckpt} has {n_ckpt} output classes but TARGET_CLASSES lists "
            f"{len(TARGET_CLASSES)}. Must match Stage 6 exactly, same order."
        )
    model = build_densenet121(len(TARGET_CLASSES)).to(DEVICE)
    model.load_state_dict(state)
    return model


# ============================================================================
# CELL 4 — DeLong's test (DeLong et al. 1988; fast algorithm Sun & Xu 2014)
# ============================================================================
def _compute_midrank(x):
    """Midranks, handling ties -- required for DeLong's structural components."""
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2


def _fast_delong(predictions_sorted_transposed, label_1_count):
    """
    Returns (aucs, covariance matrix) for k predictors on the same samples.
    Input must have all positive-class samples first.
    """
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m
    positive = predictions_sorted_transposed[:, :m]
    negative = predictions_sorted_transposed[:, m:]
    k = predictions_sorted_transposed.shape[0]

    tx = np.empty([k, m], dtype=float)
    ty = np.empty([k, n], dtype=float)
    tz = np.empty([k, m + n], dtype=float)
    for r in range(k):
        tx[r, :] = _compute_midrank(positive[r, :])
        ty[r, :] = _compute_midrank(negative[r, :])
        tz[r, :] = _compute_midrank(predictions_sorted_transposed[r, :])

    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, np.atleast_2d(delongcov)


def delong_roc_test(y_true, scores_a, scores_b):
    """
    Paired DeLong test between two score vectors on the SAME samples.
    Returns (auc_a, auc_b, z, p_two_sided).

    Valid because both arms score identical test images, so their AUCs are
    correlated; a test assuming independence would be wrong.
    """
    y_true = np.asarray(y_true).astype(int)
    order = (-y_true).argsort(kind="mergesort")   # positives first, stable
    label_1_count = int(y_true.sum())
    predictions = np.vstack((np.asarray(scores_a, dtype=float),
                             np.asarray(scores_b, dtype=float)))[:, order]

    aucs, cov = _fast_delong(predictions, label_1_count)
    var_diff = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    if var_diff <= 0:
        # Identical or near-identical predictors: difference has no variance.
        return float(aucs[0]), float(aucs[1]), 0.0, 1.0
    z = (aucs[0] - aucs[1]) / np.sqrt(var_diff)
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return float(aucs[0]), float(aucs[1]), float(z), float(p)


def bootstrap_auc_diff_ci(y_true, scores_a, scores_b, n_boot=N_BOOTSTRAP,
                           alpha=ALPHA, seed=RANDOM_SEED):
    """
    Stratified paired bootstrap CI for (AUC_a - AUC_b).

    Stratified: positives and negatives are resampled separately so every
    replicate keeps the original class balance -- with few rare-class
    positives, unstratified resampling can produce replicates with none at
    all, making AUC undefined.
    """
    y_true = np.asarray(y_true).astype(int)
    pos_idx = np.flatnonzero(y_true == 1)
    neg_idx = np.flatnonzero(y_true == 0)
    if len(pos_idx) < 2 or len(neg_idx) < 2:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(pos_idx, len(pos_idx), replace=True),
                              rng.choice(neg_idx, len(neg_idx), replace=True)])
        yb = y_true[idx]
        if yb.sum() == 0 or yb.sum() == len(yb):
            continue
        diffs.append(roc_auc_score(yb, np.asarray(scores_a)[idx])
                     - roc_auc_score(yb, np.asarray(scores_b)[idx]))
    if len(diffs) < 100:
        return np.nan, np.nan, np.nan
    diffs = np.array(diffs)
    lo, hi = np.percentile(diffs, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(np.mean(diffs)), float(lo), float(hi)


def mcnemar_test(y_true, pred_a, pred_b):
    """
    Compares the models' DECISIONS (binarised) rather than their rankings.
    Uses the exact binomial test when discordant pairs are few, where the
    chi-square approximation is unreliable.
    """
    correct_a = (np.asarray(pred_a) == np.asarray(y_true))
    correct_b = (np.asarray(pred_b) == np.asarray(y_true))
    n01 = int(np.sum(correct_a & ~correct_b))
    n10 = int(np.sum(~correct_a & correct_b))
    if n01 + n10 == 0:
        return n01, n10, 1.0
    table = [[int(np.sum(correct_a & correct_b)), n01],
             [n10, int(np.sum(~correct_a & ~correct_b))]]
    result = mcnemar(table, exact=(n01 + n10) < 25, correction=True)
    return n01, n10, float(result.pvalue)


# ============================================================================
# CELL 5 — Per-class metrics
# ============================================================================
def choose_thresholds_on_val(val_probs, val_targets, target_classes):
    """
    Pick each class's operating threshold by maximising F1 on VALIDATION.
    Choosing on test would leak the test set into the reported precision /
    recall / F1.
    """
    thresholds = {}
    for i, class_name in enumerate(target_classes):
        y, s = val_targets[:, i], val_probs[:, i]
        if y.sum() == 0 or y.sum() == len(y):
            thresholds[class_name] = 0.5
            continue
        prec, rec, thr = precision_recall_curve(y, s)
        f1 = np.divide(2 * prec * rec, prec + rec,
                       out=np.zeros_like(prec), where=(prec + rec) > 0)
        # precision_recall_curve returns len(thr) == len(prec) - 1
        best = int(np.nanargmax(f1[:-1])) if len(thr) else 0
        thresholds[class_name] = float(thr[best]) if len(thr) else 0.5
    return thresholds


def per_class_metrics(arm, probs, targets, target_classes, thresholds):
    rows = []
    for i, class_name in enumerate(target_classes):
        y, s = targets[:, i], probs[:, i]
        n_pos, n_neg = int(y.sum()), int((1 - y).sum())
        prevalence = n_pos / len(y) if len(y) else np.nan
        thr = thresholds[class_name]
        pred = (s >= thr).astype(int)

        if n_pos == 0 or n_neg == 0:
            auc = auprc = prec = rec = f1 = np.nan
        else:
            auc = roc_auc_score(y, s)
            auprc = average_precision_score(y, s)
            prec = precision_score(y, pred, zero_division=0)
            rec = recall_score(y, pred, zero_division=0)
            f1 = f1_score(y, pred, zero_division=0)

        rows.append({
            "arm": arm, "class_name": class_name,
            "n_positives": n_pos, "n_negatives": n_neg, "prevalence": prevalence,
            "roc_auc": auc, "auprc": auprc,
            "auprc_baseline": prevalence,
            "auprc_lift": (auprc / prevalence) if (prevalence and not np.isnan(auprc)) else np.nan,
            "threshold": thr, "precision": prec, "recall": rec, "f1": f1,
        })
    return pd.DataFrame(rows)


# ============================================================================
# CELL 6 — Figures
# ============================================================================
def plot_curves(all_probs, targets, target_classes, output_dir):
    for kind in ["roc", "pr"]:
        n = len(target_classes)
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 3.8))
        axes = np.atleast_1d(axes)
        for i, class_name in enumerate(target_classes):
            ax = axes[i]
            y = targets[:, i]
            for arm, probs in all_probs.items():
                s = probs[:, i]
                if y.sum() == 0 or y.sum() == len(y):
                    continue
                if kind == "roc":
                    fpr, tpr, _ = roc_curve(y, s)
                    ax.plot(fpr, tpr, label=f"{arm} ({roc_auc_score(y, s):.3f})", lw=1.4)
                else:
                    prec, rec, _ = precision_recall_curve(y, s)
                    ax.plot(rec, prec, label=f"{arm} ({average_precision_score(y, s):.3f})", lw=1.4)
            if kind == "roc":
                ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="chance")
                ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
            else:
                base = y.mean()
                ax.axhline(base, color="k", ls="--", lw=0.8, label=f"prevalence ({base:.3f})")
                ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
            ax.set_title(f"{class_name}\n(n_pos={int(y.sum())})", fontsize=9)
            ax.legend(fontsize=6); ax.grid(alpha=0.3)
        fig.suptitle("ROC by class and arm" if kind == "roc" else "Precision-Recall by class and arm")
        fig.tight_layout()
        fig.savefig(Path(output_dir) / f"stage7_{kind}_curves.png", dpi=150, bbox_inches="tight")
        plt.close(fig)


def plot_auc_bars(metrics_df, output_dir):
    pivot = metrics_df.pivot(index="class_name", columns="arm", values="roc_auc")
    pivot = pivot[[a for a in ARMS if a in pivot.columns]]
    ax = pivot.plot(kind="bar", figsize=(10, 5), width=0.8)
    ax.axhline(0.5, color="gray", ls="--", lw=1, label="chance")
    ax.set_ylabel("Test ROC-AUC"); ax.set_ylim(0, 1)
    ax.set_title("Test-set ROC-AUC by class and augmentation arm")
    ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(Path(output_dir) / "stage7_auc_by_class_bar.png", dpi=150, bbox_inches="tight")
    plt.close()


# ============================================================================
# CELL 7 — Orchestration
# ============================================================================
def main():
    out = Path(OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)

    print("[1/6] Contamination check...")
    metadata = pd.read_csv(METADATA_CSV_PATH)
    if isinstance(metadata["labels"].iloc[0], str):
        metadata["labels"] = metadata["labels"].apply(literal_eval)
    assert_no_test_contamination(metadata, GENERATED_MAP_DIRS)

    print("[2/6] Building test and validation sets...")
    labels_by_id = dict(zip(metadata["image_id"].astype(str), metadata["labels"]))
    test_ids = sorted(metadata[metadata["split"] == "test"]["image_id"].astype(str))

    # Validation split reproduced exactly as Stage 6 derived it -- used ONLY
    # to choose operating thresholds, never for reported test metrics.
    train_meta = metadata[metadata["split"] == "train"]
    rng = np.random.default_rng(RANDOM_SEED)
    real_ids = np.array(sorted(train_meta["image_id"].astype(str)))
    rng.shuffle(real_ids)
    val_ids = real_ids[:max(1, int(len(real_ids) * VAL_FRACTION))].tolist()
    print(f"      {len(test_ids)} test images | {len(val_ids)} val images (thresholds only)")

    def loader_for(ids):
        return DataLoader(EvalDataset(ids, labels_by_id, H5_PATH, TARGET_CLASSES),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

    test_loader, val_loader = loader_for(test_ids), loader_for(val_ids)

    print("[3/6] Running inference for each arm...")
    test_probs, val_probs, test_targets, val_targets = {}, {}, None, None
    for arm in ARMS:
        print(f"  {arm}")
        model = load_arm(arm)
        tp, tt = predict(model, test_loader)
        vp, vt = predict(model, val_loader)
        test_probs[arm], val_probs[arm] = tp, vp
        test_targets, val_targets = tt, vt
        # Cache raw probabilities so metrics can be recomputed later without
        # re-running the models.
        np.savez_compressed(out / f"stage7_probs_{arm}.npz",
                            test_probs=tp, test_targets=tt,
                            val_probs=vp, val_targets=vt,
                            image_ids=np.array(test_ids))
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    print("[4/6] Per-class metrics...")
    all_metrics = []
    for arm in ARMS:
        thresholds = choose_thresholds_on_val(val_probs[arm], val_targets, TARGET_CLASSES)
        all_metrics.append(per_class_metrics(arm, test_probs[arm], test_targets,
                                              TARGET_CLASSES, thresholds))
    metrics_df = pd.concat(all_metrics, ignore_index=True)
    metrics_df.to_csv(out / "stage7_per_class_metrics.csv", index=False)
    print(metrics_df[["arm", "class_name", "n_positives", "prevalence",
                       "roc_auc", "auprc", "auprc_lift", "f1"]].to_string(index=False))

    print("\n[5/6] Statistical tests...")
    test_rows = []
    for arm_a, arm_b in COMPARISON_PAIRS:
        if arm_a not in test_probs or arm_b not in test_probs:
            continue
        for i, class_name in enumerate(TARGET_CLASSES):
            y = test_targets[:, i]
            sa, sb = test_probs[arm_a][:, i], test_probs[arm_b][:, i]
            if y.sum() < 2 or (1 - y).sum() < 2:
                continue
            auc_a, auc_b, z, p_delong = delong_roc_test(y, sa, sb)
            mean_d, lo, hi = bootstrap_auc_diff_ci(y, sa, sb)

            thr_a = choose_thresholds_on_val(val_probs[arm_a], val_targets, TARGET_CLASSES)[class_name]
            thr_b = choose_thresholds_on_val(val_probs[arm_b], val_targets, TARGET_CLASSES)[class_name]
            n01, n10, p_mcnemar = mcnemar_test(y, (sa >= thr_a).astype(int), (sb >= thr_b).astype(int))

            test_rows.append({
                "comparison": f"{arm_a} vs {arm_b}", "class_name": class_name,
                "n_positives": int(y.sum()),
                "auc_a": auc_a, "auc_b": auc_b, "auc_diff": auc_a - auc_b,
                "delong_z": z, "p_delong": p_delong,
                "boot_mean_diff": mean_d, "boot_ci_low": lo, "boot_ci_high": hi,
                "ci_excludes_zero": bool(not np.isnan(lo) and (lo > 0 or hi < 0)),
                "mcnemar_n01": n01, "mcnemar_n10": n10, "p_mcnemar": p_mcnemar,
            })

    tests_df = pd.DataFrame(test_rows)
    if not tests_df.empty:
        # BH across the whole family: 4 pairs x 5 classes is 20+ tests, and
        # uncorrected p-values would overstate significance.
        for col in ["p_delong", "p_mcnemar"]:
            ok = tests_df[col].notna()
            tests_df.loc[ok, f"{col}_bh"] = multipletests(
                tests_df.loc[ok, col], alpha=ALPHA, method="fdr_bh")[1]
        tests_df.to_csv(out / "stage7_statistical_tests.csv", index=False)
        print(tests_df[["comparison", "class_name", "n_positives", "auc_diff",
                         "p_delong", "p_delong_bh", "boot_ci_low", "boot_ci_high",
                         "ci_excludes_zero"]].to_string(index=False))

    print("\n[6/6] Figures...")
    plot_curves(test_probs, test_targets, TARGET_CLASSES, out)
    plot_auc_bars(metrics_df, out)

    print(f"\nSaved to {out}")
    print("\nPRIMARY RESULT — 'filtered vs random_matched' is the causal test:")
    if not tests_df.empty:
        primary = tests_df[tests_df["comparison"] == "filtered vs random_matched"]
        if not primary.empty:
            print(primary[["class_name", "n_positives", "auc_diff",
                            "p_delong_bh", "ci_excludes_zero"]].to_string(index=False))
    print("\nReminder: one training run per arm, so these differences are not "
          "separable from seed-to-seed training noise. State that in Limitations.")
    return metrics_df, tests_df


if __name__ == "__main__":
    main()

[1/6] Contamination check...
  checked 9 directory(ies); 2205 generated image_ids
  CLEAN: 0 of 660 test images appear in any generated map.
[2/6] Building test and validation sets...
      660 test images | 461 val images (thresholds only)
[3/6] Running inference for each arm...
  none


    inference:   0%|          | 0/83 [00:00<?, ?it/s]

    inference:   0%|          | 0/58 [00:00<?, ?it/s]

  all


    inference:   0%|          | 0/83 [00:00<?, ?it/s]

...

    inference:   0%|          | 0/58 [00:00<?, ?it/s]

  random_matched


    inference:   0%|          | 0/83 [00:00<?, ?it/s]

    inference:   0%|          | 0/58 [00:00<?, ?it/s]

[4/6] Per-class metrics...
           arm    class_name  n_positives  prevalence  roc_auc    auprc  auprc_lift       f1
          none  Pneumothorax           18    0.027273 0.865395 0.559611   20.519063 0.551724
          none Consolidation           54    0.081818 0.873564 0.425639    5.202253 0.522388
          none   Nodule/Mass          117    0.177273 0.715643 0.352347    1.987600 0.384615
          none  Cardiomegaly          331    0.501515 0.934522 0.935641    1.865629 0.853372
          none   Atelectasis           27    0.040909 0.924375 0.465517   11.379311 0.421053
           all  Pneumothorax           18    0.027273 0.884432 0.399825   14.660266 0.324324
           all Consolidation           54    0.081818 0.872891 0.362188    4.426738 0.433333
           all   Nodule/Mass          117    0.177273 0.685091 0.319646    1.803130 0.385027
           all  Cardiomegaly          331    0.501515 0.937828 0.934350    1.863054 0.862119
           all   Atelectasis           27  